# Oddball — P300 ERP Analysis

Measures cortical response to rare auditory stimuli via Event-Related Potential (ERP).  
Each rare tone (2000 Hz) elicits three overlapping components analysed in separate time windows:

| Component | Window | Electrode | Expected direction | Interpretation |
| --- | --- | --- | --- | --- |
| MMN | 100–200 ms | Fz | Negative | Automatic mismatch; present even in unconscious / vegetative states |
| P3a | 200–300 ms | Cz | Positive | Automatic orienting to novelty; not specific to consciousness |
| P3b | 300–600 ms | Pz | Positive | Conscious updating — the clinically key component |

| Condition | Tone | Rate |
| --- | --- | --- |
| Standard | 1000 Hz | 80 % of tones |
| Rare (deviant) | 2000 Hz | 20 % of tones |

**Positive finding:** P3b amplitude significantly above null (p < 0.05), maximal at Pz.  
MMN and P3a serve as auditory-system integrity checks — their presence is expected automatically and is not
itself evidence of consciousness. Absence of MMN suggests a basic auditory processing problem.

> Bekinschtein et al. (2009) explicitly noted that P3a and P3b are "close in time, sometimes difficult to
> differentiate," and designed the global-local paradigm specifically to circumvent that ambiguity. Our
> simple 2-tone oddball cannot fully separate them — the Pz/300–600 ms focus and the parietal topography
> check are our primary guards against P3a contamination.

## 1. Configuration

**How to run this notebook:**

1. Run **cell 1a** — a session dropdown appears. Pick the patient and date. Optionally list bad channels (comma-separated).
2. Run **cell 1b** — reads your selection and sets up all file paths. Check the printed output looks correct.
3. Click on cell 1b, then choose **Run → Run Selected Cell and All Below** from the menu to execute the rest of the notebook in one go.

> To switch patients mid-session: change the dropdown, re-run cell 1b, then use **Run → Run Selected Cell and All Below** again.

In [1]:
import sys
import re
from pathlib import Path
import ipywidgets as widgets
from IPython.display import display

ANALYSIS_ROOT = Path.cwd().resolve()
if ANALYSIS_ROOT.name != 'analysis':
    ANALYSIS_ROOT = next(
        (parent for parent in [ANALYSIS_ROOT, *ANALYSIS_ROOT.parents]
         if parent.name == 'analysis' and (parent / 'notebooks').exists()),
        ANALYSIS_ROOT,
    )
if str(ANALYSIS_ROOT) not in sys.path:
    sys.path.insert(0, str(ANALYSIS_ROOT))

from lib.io import DEFAULT_EEG_CHANNELS

repo_root = ANALYSIS_ROOT.parent
csv_dir = repo_root / 'stimulus_software' / 'patient_data' / 'results'
edf_dir = repo_root / 'stimulus_software' / 'patient_data' / 'edfs'

edf_by_patient = {
    f.name.replace('_clipped.EDF', '').replace('_clipped.edf', ''): f
    for f in sorted(edf_dir.glob('*.[Ee][Dd][Ff]'))
    if '_clipped' in f.name
}

def _parse_csv(path):
    m = re.search(r'(\d{4}-\d{2}-\d{2})_stimulus_results', path.stem)
    if not m:
        return None
    date = m.group(1)
    patient_id = path.stem[:path.stem.index('_' + date)]
    return patient_id, date

sessions = []
for f in sorted(csv_dir.glob('*_stimulus_results.csv')):
    parsed = _parse_csv(f)
    if parsed is None:
        continue
    pid, date = parsed
    sessions.append({'csv': f, 'patient_id': pid, 'date': date, 'edf': edf_by_patient.get(pid)})

options = [(f'{s["patient_id"]}  —  {s["date"]}', i) for i, s in enumerate(sessions)]

csv_dropdown = widgets.Dropdown(
    options=options,
    description='Session:',
    layout=widgets.Layout(width='360px'),
    style={'description_width': 'initial'},
)
edf_status = widgets.Label()

def _on_change(_):
    s = sessions[csv_dropdown.value]
    edf_status.value = f'EDF: {s["edf"].name}' if s['edf'] else '⚠  No matching EDF found for this patient'

csv_dropdown.observe(_on_change, names='value')
_on_change(None)

display(csv_dropdown, edf_status)
print('Select a session above, then run the next cell.')

Dropdown(description='Session:', layout=Layout(width='360px'), options=(('CON010  —  2026-03-06', 0), ('CON011…

Label(value='EDF: CON010_clipped.EDF')

Select a session above, then run the next cell.


### 1b. Confirm selection — run after choosing from the dropdown above

In [2]:
_sel = sessions[csv_dropdown.value]
SUBJECT_ID   = _sel['patient_id']
SESSION_DATE = _sel['date']
CSV_PATH     = _sel['csv']
EDF_PATH     = _sel['edf']

if EDF_PATH is None or not EDF_PATH.exists():
    raise FileNotFoundError(
        f'No EDF found for {SUBJECT_ID} — '
        f'add {SUBJECT_ID}_clipped.EDF to patient_data/edfs/'
    )

OUT_DIR = ANALYSIS_ROOT / 'results' / SUBJECT_ID / 'oddball'
OUT_DIR.mkdir(parents=True, exist_ok=True)

EEG_CHANNELS = DEFAULT_EEG_CHANNELS.copy()
BAD_CHANNELS = []

print(f'Subject:    {SUBJECT_ID}')
print(f'Date:       {SESSION_DATE}')
print(f'EDF:        {EDF_PATH.name}')
print(f'CSV:        {CSV_PATH.name}')
print(f'Output dir: {OUT_DIR}')

Subject:    CON013
Date:       2026-04-10
EDF:        CON013_clipped.EDF
CSV:        CON013_2026-04-10_stimulus_results.csv
Output dir: /Users/joey/Documents/EEG Project/code/analysis/results/CON013/oddball


## 2. Imports

In [3]:
import numpy as np
import pandas as pd
import mne
import matplotlib.pyplot as plt
from scipy import stats

from lib.io import align_stimulus_csv, load_raw_eeg_metadata
from lib.preprocessing import load_filtered_eeg

%matplotlib inline
mne.set_log_level('WARNING')
print(f'MNE {mne.__version__}')

MNE 1.11.0


## 3. Load EDF + Sync Alignment

In [4]:
raw, sfreq, available_eeg = load_raw_eeg_metadata(
    EDF_PATH,
    eeg_channels=EEG_CHANNELS,
    bad_channels=BAD_CHANNELS,
    preload=False,
    verbose=False,
)
print(f'EEG channels ({len(available_eeg)}): {available_eeg}')
print(f'sfreq: {sfreq} Hz  |  duration: {raw.n_times/sfreq:.0f}s')

EEG channels (19): ['Fp1', 'Fp2', 'Fz', 'F3', 'F4', 'F7', 'F8', 'Cz', 'C3', 'C4', 'T3', 'T4', 'T5', 'T6', 'Pz', 'P3', 'P4', 'O1', 'O2']
sfreq: 512.0 Hz  |  duration: 4846s


/Users/joey/Documents/EEG Project/code/analysis/lib/io.py:93: RuntimeWarning: Omitted 5 annotation(s) that were outside data range.
  raw = mne.io.read_raw_edf(edf_path, preload=preload, verbose=verbose)
/Users/joey/Documents/EEG Project/code/analysis/lib/io.py:102: RuntimeWarning: The unit for channel(s) DC10, DC2, DC3, DC4, DC5, DC6, DC7, DC8, DC9 has changed from V to NA.
  raw.set_channel_types({ch: "misc" for ch in dc_channels})


In [5]:
df, time_offset = align_stimulus_csv(CSV_PATH, sfreq=sfreq, n_times=raw.n_times)
print(f'time_offset = {time_offset:.4f} s')
print(f'sync rows: {(df['stim_type'] == 'sync_detection').sum()} detection, {(df['stim_type'] == 'manual_sync_pulse').sum()} manual pulse')

time_offset = -1828.7082 s
sync rows: 1 detection, 1 manual pulse


## 4. Preprocessing

In [ ]:
# 0.1 Hz highpass (not 1 Hz) — preserves slow ERP components like MMN (100-200 ms)
# Average reference — removes bias from the recording reference electrode
print(f'Loading {len(available_eeg)} EEG channels from {EDF_PATH.name}...')
raw_p300 = load_filtered_eeg(raw, available_eeg, l_freq=0.1, h_freq=30, verbose=False)
raw_p300.set_eeg_reference('average', projection=False, verbose=False)
print('Preprocessing done.')

## 5. Epoch

In [ ]:
odd_df = df[df['stim_type'].str.startswith('oddball')].copy()
rare_df = odd_df[odd_df['notes'] == 'rare_tone']
std_df  = odd_df[odd_df['notes'] == 'standard_tone']

print(f'Oddball tones: {len(odd_df)} total  ({len(rare_df)} rare, {len(std_df)} standard)')

if len(rare_df) == 0:
    raise RuntimeError(
        'No per-beep oddball rows found. '
        'This subject may predate per-beep CSV logging. '
        'Check stim_type values with: df["stim_type"].value_counts()'
    )

intervals = np.diff(odd_df['edf_start'].sort_values().values)
print(f'Tone interval: mean={intervals.mean():.4f}s  std={intervals.std():.6f}s  (expected 1.0s)')

n_rare_pre = len(rare_df)
n_std_pre  = len(std_df)

rare_events = np.column_stack([
    rare_df['start_sample'].values,
    np.zeros(len(rare_df), dtype=int),
    np.full(len(rare_df), 2, dtype=int)
])
std_events = np.column_stack([
    std_df['start_sample'].values,
    np.zeros(len(std_df), dtype=int),
    np.full(len(std_df), 1, dtype=int)
])
all_events = np.vstack([rare_events, std_events])
all_events = all_events[all_events[:, 0].argsort()]

REJECT_THRESHOLD_UV = 200
epochs = mne.Epochs(
    raw_p300, events=all_events,
    event_id={'standard': 1, 'rare': 2},
    tmin=-0.2, tmax=0.8, baseline=(-0.2, 0),
    reject=dict(eeg=REJECT_THRESHOLD_UV * 1e-6),
    preload=True, verbose=False
)

n_rare_post = len(epochs['rare'])
n_std_post  = len(epochs['standard'])
print(f'Epochs kept: {len(epochs)}  ({n_rare_post} rare, {n_std_post} std)')
print(f'Rejected:    {n_rare_pre - n_rare_post} rare, {n_std_pre - n_std_post} std  '
      f'(threshold: {REJECT_THRESHOLD_UV} µV)')

## 6. Compute ERPs

In [ ]:
evoked_rare = epochs['rare'].average()
evoked_std  = epochs['standard'].average()
diff_evoked = mne.combine_evoked([evoked_rare, evoked_std], weights=[1, -1])

# One electrode per component — the published primary channel for each
COMPONENTS = {
    'MMN': {'win': (0.100, 0.200), 'ch': 'Fz', 'sign': -1, 'label': 'Automatic mismatch'},
    'P3a': {'win': (0.200, 0.300), 'ch': 'Cz', 'sign': +1, 'label': 'Automatic orienting'},
    'P3b': {'win': (0.300, 0.600), 'ch': 'Pz', 'sign': +1, 'label': 'Conscious updating (P300)'},
}

t = diff_evoked.times
print('Rare − Standard mean amplitude by component window:')
for name, comp in COMPONENTS.items():
    ch = comp['ch']
    if ch not in diff_evoked.ch_names:
        print(f'  {name} ({ch}): channel not available')
        continue
    win_mask = (t >= comp['win'][0]) & (t <= comp['win'][1])
    idx = diff_evoked.ch_names.index(ch)
    amp = diff_evoked.data[idx][win_mask].mean() * 1e6
    expected = 'positive' if comp['sign'] > 0 else 'negative'
    print(f'  {name}  {int(comp["win"][0]*1000)}-{int(comp["win"][1]*1000)} ms  {ch}: {amp:+.2f} µV  (expected {expected})')

## 7. Permutation Test

Each component window is tested separately with a one-sided permutation test (1,000 shuffles).  
Expected directions: MMN negative at Fz, P3a positive at Cz, P3b positive at Pz.

Bonferroni threshold for 3 simultaneous tests: **p < 0.017**.  
The P3b result is the primary clinical finding; MMN and P3a are auditory-system integrity checks.

In [ ]:
N_PERMS     = 1000
epochs_data = epochs.get_data()   # (n_tones, n_ch, n_times)
labels      = epochs.events[:, 2]
t_ep        = epochs.times
# Each component gets its own seeded RNG so results are reproducible
# regardless of iteration order. P3b uses seed 42 (matches original).
COMPONENT_SEEDS = {'MMN': 40, 'P3a': 41, 'P3b': 42}

perm_results = {}
print(f'Running {N_PERMS} permutations per component window...\n')

for name, comp in COMPONENTS.items():
    ch = comp['ch']
    if ch not in epochs.ch_names:
        print(f'{name}: channel {ch} not available — skipping')
        continue
    ch_idx   = epochs.ch_names.index(ch)
    win_mask = (t_ep >= comp['win'][0]) & (t_ep <= comp['win'][1])
    sign     = comp['sign']
    rng      = np.random.default_rng(COMPONENT_SEEDS.get(name, 42))

    def _amp(data, labs, _ch=ch_idx, _win=win_mask):
        rare_mean = data[labs == 2, _ch][:, _win].mean()
        std_mean  = data[labs == 1, _ch][:, _win].mean()
        return (rare_mean - std_mean) * 1e6

    obs  = _amp(epochs_data, labels)
    null = np.array([_amp(epochs_data, rng.permutation(labels)) for _ in range(N_PERMS)])
    p    = np.mean(null <= obs) if sign < 0 else np.mean(null >= obs)

    perm_results[name] = {'obs': obs, 'null': null, 'p': p, 'ch': ch, 'comp': comp}
    print(f'{name}  {int(comp["win"][0]*1000)}-{int(comp["win"][1]*1000)} ms  {ch}:')
    print(f'  Observed: {obs:+.3f} uV  |  p = {p:.4f}')
    print()

bonf = 0.05 / len(perm_results)
print(f'Bonferroni threshold for {len(perm_results)} tests: p < {bonf:.4f}')

# Backwards-compat aliases (P3b is the primary clinical result)
_p3b         = perm_results.get('P3b', {})
observed_amp = _p3b.get('obs', float('nan'))
null         = _p3b.get('null', np.array([]))
p_val        = _p3b.get('p', float('nan'))
P300_CH      = 'Pz'

## 8. Plots

In [ ]:
times_ms = diff_evoked.times * 1000

PLOT_CHANNELS = list(dict.fromkeys(
    comp['ch'] for comp in COMPONENTS.values()
    if comp['ch'] in diff_evoked.ch_names
))  # [Fz, Cz, Pz] — deduped, preserves order

WINDOW_STYLE = {
    'MMN': {'color': 'lightblue',  'alpha': 0.22, 'label': 'MMN (100-200 ms)'},
    'P3a': {'color': 'lightgreen', 'alpha': 0.22, 'label': 'P3a (200-300 ms)'},
    'P3b': {'color': 'gold',       'alpha': 0.22, 'label': 'P3b / P300 (300-600 ms)'},
}

fig, axes = plt.subplots(len(PLOT_CHANNELS), 1,
                         figsize=(10, 3 * len(PLOT_CHANNELS)), sharex=True)
if len(PLOT_CHANNELS) == 1:
    axes = [axes]

for ax, ch_name in zip(axes, PLOT_CHANNELS):
    idx     = diff_evoked.ch_names.index(ch_name)
    rare_uv = evoked_rare.data[idx] * 1e6
    std_uv  = evoked_std.data[idx]  * 1e6
    diff_uv = rare_uv - std_uv

    ax.plot(times_ms, std_uv,  color='steelblue', lw=1.5, label='Standard', alpha=0.85)
    ax.plot(times_ms, rare_uv, color='firebrick',  lw=1.5, label='Rare',     alpha=0.85)
    ax.plot(times_ms, diff_uv, color='darkgreen',  lw=1.5, ls='--', label='Rare - Std')
    ax.axvline(0, color='k', lw=0.8, ls=':')
    ax.axhline(0, color='k', lw=0.5)

    for name, ws in WINDOW_STYLE.items():
        comp = COMPONENTS[name]
        ax.axvspan(comp['win'][0] * 1000, comp['win'][1] * 1000,
                   color=ws['color'], alpha=ws['alpha'], label=ws['label'])

    comp_name = next((n for n, c in COMPONENTS.items() if c['ch'] == ch_name), None)
    if comp_name and comp_name in perm_results:
        p = perm_results[comp_name]['p']
        ax.set_title(
            f'{ch_name}  —  {COMPONENTS[comp_name]["label"]}  (p={p:.3f})',
            fontsize=10
        )
    else:
        ax.set_title(ch_name, fontsize=10)

    ax.set_ylabel('µV')
    ax.legend(loc='upper right', fontsize=8)
    ax.grid(True, alpha=0.3)

axes[-1].set_xlabel('Time (ms)')
fig.suptitle(f'{SUBJECT_ID} — Oddball ERP: Rare vs Standard', fontsize=13)
plt.tight_layout()
plt.savefig(OUT_DIR / f'{SUBJECT_ID}_oddball_p300.png', dpi=150)
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))

highlight = {'Fz': 'royalblue', 'Cz': 'forestgreen', 'Pz': 'firebrick'}

for ch_name in diff_evoked.ch_names:
    idx = diff_evoked.ch_names.index(ch_name)
    y   = diff_evoked.data[idx] * 1e6
    if ch_name in highlight:
        ax.plot(times_ms, y, color=highlight[ch_name], lw=2.0, zorder=3, label=ch_name)
    else:
        ax.plot(times_ms, y, color='#aaaaaa', lw=0.9, zorder=1, alpha=0.9)

for name, comp in COMPONENTS.items():
    ws = {'MMN': 'lightblue', 'P3a': 'lightgreen', 'P3b': 'gold'}[name]
    ax.axvspan(comp['win'][0]*1000, comp['win'][1]*1000, color=ws, alpha=0.2)

ax.axvline(0, color='k', lw=0.8, ls=':')
ax.axhline(0, color='k', lw=0.5)
ax.set(xlabel='Time (ms)', ylabel='Rare - Standard (uV)',
       title=f'{SUBJECT_ID} — Rare minus Standard: all electrodes')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(OUT_DIR / f'{SUBJECT_ID}_oddball_butterfly.png', dpi=150)
plt.show()

In [ ]:
n_comp = len(perm_results)
fig, axes = plt.subplots(n_comp, 1, figsize=(10, 4 * n_comp))
if n_comp == 1:
    axes = [axes]

for ax, (name, res) in zip(axes, perm_results.items()):
    sign     = res['comp']['sign']
    null_arr = res['null']
    obs      = res['obs']
    p        = res['p']
    pct_label = '5th pct' if sign < 0 else '95th pct'
    pct_val   = np.percentile(null_arr, 5 if sign < 0 else 95)

    ax.hist(null_arr, bins=40, color='steelblue', alpha=0.7, label='Null distribution')
    ax.axvline(obs,     color='firebrick', lw=2,       label=f'Observed: {obs:.2f} µV')
    ax.axvline(pct_val, color='k',         lw=1, ls='--', label=pct_label)
    ax.set(xlabel='Rare - Standard amplitude (µV)', ylabel='Count',
           title=f'{name}  at {res["ch"]}  —  p = {p:.3f}')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

fig.suptitle(
    f'Permutation null distributions  (Bonferroni threshold: p < {0.05/n_comp:.3f})',
    fontsize=11
)
plt.tight_layout()
plt.savefig(OUT_DIR / f'{SUBJECT_ID}_p300_null.png', dpi=150)
plt.show()

In [ ]:
montage = mne.channels.make_standard_montage('standard_1020')
diff_evoked.set_montage(montage, match_case=False, on_missing='warn')

fig = diff_evoked.plot_topomap(
    times=[0.1, 0.2, 0.3, 0.4, 0.5, 0.6], scalings=dict(eeg=1e6),
    show=False, cmap='RdBu_r'
)
fig.suptitle(
    f'{SUBJECT_ID} — Rare − Standard topomap (µV)\n'
    '100–200 ms: MMN  |  200–300 ms: P3a  |  300–600 ms: P3b',
    fontsize=11, y=1.02
)
fig.savefig(OUT_DIR / f'{SUBJECT_ID}_p300_topomap.png', dpi=150, bbox_inches='tight')
plt.show()

## 9. Johnsen Band-Power Reactivity

Quantitative reactivity analysis inspired by Johnsen et al.: compare **log band power**
in pre-stimulus (reference) vs post-stimulus (active) epochs across four EEG frequency bands.

| Band | Range | Expected in reactive brain |
| --- | --- | --- |
| Delta | 1–3 Hz | May increase with stimulus processing |
| Theta | 4–7 Hz | Associated with cognitive engagement |
| Alpha | 8–13 Hz | Often suppresses (ERD) after auditory stimulus |
| Beta | 14–30 Hz | Context-dependent reactivity |

Z-scores > ±1.96 indicate significant departure from baseline (p ≈ 0.05, two-sided).

In [ ]:
FREQ_BANDS = {
    'delta': (1, 3),
    'theta': (4, 7),
    'alpha': (8, 13),
    'beta':  (14, 30),
}
REF_TMIN, REF_TMAX =  -2.0, -0.05   # pre-stimulus reference window
ACT_TMIN, ACT_TMAX =   0.0,  2.0    # post-stimulus active window

ref_epochs_j = mne.Epochs(
    raw_p300, events=all_events, event_id={'standard': 1, 'rare': 2},
    tmin=REF_TMIN, tmax=REF_TMAX, baseline=None, preload=True, verbose=False
)
act_epochs_j = mne.Epochs(
    raw_p300, events=all_events, event_id={'standard': 1, 'rare': 2},
    tmin=ACT_TMIN, tmax=ACT_TMAX, baseline=None, preload=True, verbose=False
)
print(f'Reference epochs: {len(ref_epochs_j)}  |  Active epochs: {len(act_epochs_j)}')

In [ ]:
# n_fft capped at epoch length — ref window is -2.0 to -0.05 s (1.95 s), shorter than 2 s
n_fft_j = min(int(sfreq * 2), ref_epochs_j.get_data().shape[-1])
ref_psd_j = ref_epochs_j.compute_psd(method='welch', n_fft=n_fft_j,
                                       n_overlap=n_fft_j // 2,
                                       fmin=1, fmax=30, verbose=False)
act_psd_j = act_epochs_j.compute_psd(method='welch', n_fft=n_fft_j,
                                       n_overlap=n_fft_j // 2,
                                       fmin=1, fmax=30, verbose=False)
print(f'PSD shape: {ref_psd_j.shape}  (n_epochs, n_ch, n_freqs)')

In [ ]:
def log_band_power(psd_obj, bands):
    """Returns dict band -> np.array (n_epochs,): log mean power across channels and band."""
    data = psd_obj.get_data()   # (n_epochs, n_ch, n_freqs)
    freqs = psd_obj.freqs
    result = {}
    for band, (flo, fhi) in bands.items():
        idx = np.where((freqs >= flo) & (freqs <= fhi))[0]
        band_avg = data[:, :, idx].mean(axis=(1, 2))   # (n_epochs,)
        result[band] = np.log(band_avg + 1e-30)
    return result

ref_log_j = log_band_power(ref_psd_j, FREQ_BANDS)
act_log_j = log_band_power(act_psd_j, FREQ_BANDS)

# Z-score: active epochs relative to reference baseline
z_scores_j = {}
for band in FREQ_BANDS:
    ref_mean = ref_log_j[band].mean()
    ref_std  = ref_log_j[band].std() + 1e-30
    z_scores_j[band] = (act_log_j[band] - ref_mean) / ref_std

print(f'Johnsen Reactivity Summary (Z-score: active vs pre-stimulus reference):')
print(f'{"Band":<8} {"Mean Z":>10} {"Max |Z|":>10} {"Sig epochs (|Z|>1.96)":>24}')
print('-' * 56)
for band, zs in z_scores_j.items():
    sig = int(np.sum(np.abs(zs) > 1.96))
    print(f'{band:<8} {zs.mean():>10.3f} {np.abs(zs).max():>10.3f} {sig:>22}/{len(zs)}')

In [ ]:
colors = ['steelblue', 'darkorange', 'forestgreen', 'mediumpurple']
fig, axes = plt.subplots(2, 2, figsize=(12, 7))
axes = axes.flatten()

for ax, (band, zs), color in zip(axes, z_scores_j.items(), colors):
    flo, fhi = FREQ_BANDS[band]
    t_idx = np.arange(len(zs))
    ax.plot(t_idx, zs, color=color, marker='o', ms=4, lw=1.2)
    ax.axhline( 1.96, color='green', ls='--', lw=0.9, label='+1.96 (p≈0.05)')
    ax.axhline(-1.96, color='red',   ls='--', lw=0.9, label='-1.96 (p≈0.05)')
    ax.axhline(0, color='k', lw=0.5)
    sig_up   = zs >  1.96
    sig_down = zs < -1.96
    if sig_up.any():   ax.scatter(t_idx[sig_up],   zs[sig_up],   color='green', zorder=5, s=50)
    if sig_down.any(): ax.scatter(t_idx[sig_down], zs[sig_down], color='red',   zorder=5, s=50)
    ax.set(title=f'{band.capitalize()}  ({flo}–{fhi} Hz)',
           xlabel='Epoch index', ylabel='Z-score')
    ax.grid(True, alpha=0.3)

handles, labels_leg = axes[0].get_legend_handles_labels()
fig.legend(handles, labels_leg, loc='lower center', ncol=2, fontsize=9,
           bbox_to_anchor=(0.5, 0.01))
fig.suptitle(f'{SUBJECT_ID} — Johnsen Band-Power Reactivity (post vs pre-stimulus)', fontsize=12)
plt.tight_layout(rect=[0, 0.06, 1, 1])
plt.savefig(OUT_DIR / f'{SUBJECT_ID}_oddball_johnsen_reactivity.png', dpi=150)
plt.show()

## 10. Summary

**Three-component result:**

| Component | Window | Electrode | Significant? | Clinical meaning |
| --- | --- | --- | --- | --- |
| MMN | 100–200 ms | Fz | negative p < 0.05? | Auditory cortex working (automatic, not consciousness-specific) |
| P3a | 200–300 ms | Cz | positive p < 0.05? | Automatic orienting intact (expected, not consciousness-specific) |
| **P3b** | **300–600 ms** | **Pz** | **positive p < 0.05?** | **Evidence of conscious processing — primary clinical finding** |

**Interpretation:**
- MMN only: basic auditory processing intact, no conscious response detected
- MMN + P3a, no P3b: automatic orienting intact, still no conscious response
- P3b significant (p < 0.05): preserved cortical evaluation of deviant stimuli — consistent with awareness
- Bonferroni threshold for strict correction: p < 0.017

**Caveat:** In a simple 2-tone oddball, P3a and P3b cannot be perfectly separated (both are centrally
positive). Pz focus and parietal topography are the primary guards against P3a contamination. A genuine
P3b should show a parietal maximum in the 300–600 ms topomap with simultaneous frontal negativity.

**Johnsen reactivity:** Z > ±1.96 in one or more frequency bands indicates measurable power modulation.